# CSC2107: Design & Analysis of Algorithms
## Assignment 5.1 — Naive vs. Memoized Fibonacci


**Objective:** Implement plain recursive Fibonacci and recursive Fibonacci with memoization, compare their function-call counts and execution times, plot the results, and analyze their complexity.



## Part 1 — Implementation

The Fibonacci sequence is defined by:

- \(F(0)=0\)
- \(F(1)=1\)
- \(F(n)=F(n-1)+F(n-2)\), for \(n\geq2\)

Algorithm A uses plain recursion. Algorithm B uses a dictionary as a memoization table to reuse values already calculated.


In [1]:
import time
import sys
import csv
import math
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

print("Python recursion limit:", sys.getrecursionlimit())


Python recursion limit: 1000


### Algorithm A: Plain recursion

The global counter `calls_a` increases every time the function is entered, including calls that return the base cases.


In [ ]:
calls_a = 0

def fib_naive(n):
    global calls_a
    calls_a += 1

    if n == 0 or n == 1:
        return n

    return fib_naive(n - 1) + fib_naive(n - 2)


In [ ]:
plt.figure(figsize=(9, 5.5))

plt.plot(
    naive_df["n"],
    naive_df["calls"],
    marker="o",
    label="Algorithm A: Naive"
)

plt.yscale("log")
plt.xlabel("n")
plt.ylabel("Number of function calls (log scale)")
plt.title("Algorithm A: Naive Fibonacci")
plt.grid(True, which="both")
plt.legend()
plt.show()

### Algorithm B: Recursion with memoization

The dictionary `fibtable` stores computed Fibonacci values. The test `n in fibtable` checks whether the key exists, even if the stored value is zero.


In [ ]:
calls_b = 0
fibtable = {}

def fib_memo(n):
    global calls_b
    calls_b += 1

    if n in fibtable:
        return fibtable[n]

    if n == 0 or n == 1:
        value = n
    else:
        value = fib_memo(n - 1) + fib_memo(n - 2)

    fibtable[n] = value
    return value


### Check correctness for n = 0 to 20

The assertion stops the cell if the two algorithms return different results.


In [ ]:
correctness_results = []

for n in range(21):
    calls_a = 0
    result_a = fib_naive(n)

    calls_b = 0
    fibtable = {}
    result_b = fib_memo(n)

    assert result_a == result_b, f"Mismatch at n={n}"
    correctness_results.append({
        "n": n,
        "F(n)": result_a,
        "Naive calls": calls_a,
        "Memoized calls": calls_b
    })

display(pd.DataFrame(correctness_results))
print("Correctness test passed: both algorithms agree for n = 0 to 20.")


## Part 2 — Experiments

For each run, measure elapsed time with `time.perf_counter()` and count function calls. The memoization table is cleared before every memoized run. The naive experiment stops if a run takes longer than 60 seconds.


### Algorithm A: Naive recursion

Requested inputs: 5, 10, 15, 20, 25, 30, and 35. If a run exceeds 60 seconds, it is recorded and the experiment stops.


In [ ]:
naive_rows = []

for n in [5, 10, 15, 20, 25, 30, 35]:
    calls_a = 0
    start = time.perf_counter()
    value = fib_naive(n)
    elapsed = time.perf_counter() - start

    naive_rows.append({
        "algorithm": "Naive",
        "n": n,
        "calls": calls_a,
        "time (s)": elapsed,
        "F(n)": value
    })

    print(f"n={n:>2} | calls={calls_a:>12,} | time={elapsed:.6f} s")

    if elapsed > 60:
        print("A run exceeded 60 seconds. Stopping the naive experiment.")
        break

naive_df = pd.DataFrame(naive_rows)
display(naive_df)


### Algorithm B: Memoized recursion

Requested inputs: 10, 50, 100, 200, 500, and 1000. A `RecursionError` is recorded if the current Python recursion limit prevents a run from completing.


In [ ]:
memo_rows = []

for n in [10, 50, 100, 200, 500, 1000]:
    calls_b = 0
    fibtable = {}  # Important: clear the cache before each run

    start = time.perf_counter()
    try:
        value = fib_memo(n)
        status = "Completed"
    except RecursionError:
        value = None
        status = "RecursionError"
    elapsed = time.perf_counter() - start

    memo_rows.append({
        "algorithm": "Memoized",
        "n": n,
        "calls": calls_b,
        "time (s)": elapsed,
        "F(n)": value,
        "status": status
    })

    print(f"n={n:>4} | calls={calls_b:>6,} | time={elapsed:.6f} s | {status}")

memo_df = pd.DataFrame(memo_rows)
display(memo_df)


### Combined results





In [ ]:
all_results = pd.concat([naive_df, memo_df], ignore_index=True, sort=False)
display(all_results)

all_results.to_csv("fibonacci_results.csv", index=False)
print("Saved fibonacci_results.csv")


### Graph: n against number of function calls

A logarithmic y-axis is used because the naive algorithm's call count grows exponentially. This makes it possible to view the much smaller memoized call counts on the same graph.


In [ ]:
plt.figure(figsize=(9, 5.5))

plt.plot(naive_df["n"], naive_df["calls"], marker="o", label="Naive recursion")
plt.plot(memo_df["n"], memo_df["calls"], marker="s", label="Memoized recursion")

plt.yscale("log")
plt.xlabel("n (Fibonacci index)")
plt.ylabel("Number of function calls (log scale)")
plt.title("Naive vs. Memoized Fibonacci")
plt.grid(True, which="both", linestyle="--", linewidth=0.5)
plt.legend()
plt.tight_layout()
plt.savefig("fibonacci_calls.png", dpi=200)
plt.show()

print("Saved fibonacci_calls.png")


## Part 3 — Analysis




In [ ]:
# Estimate naive calls using exact integer arithmetic.
# This computes the Fibonacci number iteratively; it does NOT run the naive recursion.
def fib_iterative(n):
    if n < 2:
        return n
    previous, current = 0, 1
    for _ in range(2, n + 1):
        previous, current = current, previous + current
    return current

estimated_calls_1000 = 2 * fib_iterative(1001) - 1
print("Digits in the estimated call count:", len(str(estimated_calls_1000)))
print("Scientific-notation estimate: {:.3e}".format(estimated_calls_1000))

# Estimate using the measured call rate from the largest completed naive run.
largest_naive = naive_df.iloc[-1]
measured_rate = largest_naive["calls"] / largest_naive["time (s)"]
estimated_seconds = estimated_calls_1000 / measured_rate

print(f"Measured call rate: {measured_rate:,.2f} calls/second")
print("Estimated seconds for naive n=1000: {:.3e}".format(estimated_seconds))


## Part 4 — Pushing to n = 1000

### 4.1 Try memoized recursion at n = 1000 with the default settings

Python's recursion limit can be checked with `sys.getrecursionlimit()`. The exact result may vary by Python version and environment.


In [ ]:
print("Current recursion limit:", sys.getrecursionlimit())

calls_b = 0
fibtable = {}
try:
    result_1000_recursive = fib_memo(1000)
    print("Memoized recursion completed.")
    print("Number of digits in F(1000):", len(str(result_1000_recursive)))
except RecursionError as error:
    result_1000_recursive = None
    print(type(error).__name__ + ":", error)


### 4.2 Fix 1 — Increase the recursion limit

Increasing the limit may let the recursive version finish, but setting it too high can risk exhausting stack resources. Restore the old setting afterward.


In [ ]:
old_limit = sys.getrecursionlimit()

try:
    sys.setrecursionlimit(max(old_limit, 3000))
    calls_b = 0
    fibtable = {}
    result_1000_higher_limit = fib_memo(1000)
    print("Completed with higher recursion limit.")
    print("Number of digits in F(1000):", len(str(result_1000_higher_limit)))
finally:
    sys.setrecursionlimit(old_limit)
    print("Restored recursion limit to:", sys.getrecursionlimit())


**Trade-off:** This keeps the recursive memoization approach, but relies on a deeper call stack and may not be safe with an excessively large recursion limit.

### 4.3 Fix 2 — Iterative dynamic programming

This version uses a loop instead of recursive calls and requires only two running Fibonacci values.


In [ ]:
def fib_iterative(n):
    if n < 2:
        return n

    previous, current = 0, 1
    for _ in range(2, n + 1):
        previous, current = current, previous + current

    return current

result_1000_iterative = fib_iterative(1000)
print("F(1000) has", len(str(result_1000_iterative)), "digits.")
assert result_1000_iterative == fib_iterative(1000)


**Trade-off:** Iteration avoids recursion-depth errors and uses \(O(1)\) auxiliary space, but it is an iterative dynamic-programming solution rather than the recursive memoization algorithm.

### 4.4 The memoization-table check

If a table is a list filled with zeros and the condition is `if fibtable[n]:`, a stored value of zero is treated as false. For example, \(F(0)=0\), so the test can miss an already-stored answer.

Use `if n in fibtable:` for a dictionary, or `if fibtable[n] is not None:` for a list initialized with `None`. This issue can cause unnecessary recalculation, but a correct existence check does not change the intended asymptotic complexity.


## Part 5 — Conclusion

The experiment compared plain recursive Fibonacci with recursive Fibonacci using memoization. Both algorithms returned the same Fibonacci values for inputs from n = 0 to 20. However, their execution times and function-call counts increased at very different rates.

For Algorithm A, the measured execution time increased from approximately 0.000005 seconds at n = 5 to 4.521093 seconds at n = 35. Over the same inputs, the number of calls rose from 15 to 29,860,703. This sharp increase demonstrates the cost of repeatedly solving the same subproblems. Its time complexity is exponential, Θ(φⁿ), where φ ≈ 1.618, so using naive recursion for n = 1000 is not feasible. Based on the measured call rate, the notebook estimated that it could take around 2.13 × 10²⁰² seconds.

Algorithm B was much faster for the tested inputs. Its measured time was approximately 0.000013 seconds at n = 10 and 0.000508 seconds at n = 500, with 999 function calls at n = 500. This supports the linear time complexity, Θ(n), expected when each Fibonacci value is computed once and stored for reuse. The first attempt at n = 1000 raised a RecursionError because Python's default recursion limit was 1000; after increasing the limit for the test, the recursive memoized algorithm completed.

Overall, memoization was substantially more efficient because it avoids recalculating overlapping subproblems. The measured times are specific to this notebook run and computer, so they may differ on another system. The main lesson is that algorithm design—not just the speed of the computer—strongly affects performance as input size grows.